# Computer Vision Models on Hugging Face

**Companion lab for MIT 15.773 Hands-On Deep Learning (Spring 2024).**

This notebook is adapted from `11c_cv_models_huggingface.ipynb` for study and reproducibility. The instructional cells and code are preserved, while stored outputs are removed to keep the book portable. Run cells in order and inspect shapes, metrics, and failure cases rather than treating successful execution as the only goal.

> Some labs require a GPU, external datasets, model downloads, or API credentials. Use a hosted runtime where the notebook indicates one; never commit credentials.


**Credit**: Adapted from [here](https://github.com/NielsRogge/Transformers-Tutorials/blob/master/HuggingFace_vision_ecosystem_overview_(June_2022).ipynb).

# Computer Vision Models on the HuggingFace Hub

In practice, if you have a CV problem that neatly maps to a standard CV task (e.g., image classification, object detection, segmentation), we recommend looking for a task-relevant CV model on the Hugging Face 🤗 [hub](https://huggingface.co/). If you find one, you can start using it immediately. In this notebook, we will show you how to do this.

We will first start with a quick tour of the 🤗 hub and then continue with this notebook.



## Set-up environment

The `transformers` library from 🤗 is the key Python package we will be using. Happily, this library comes preinstalled with Colab.

However, for the `object detection` and `image segmentation` examples we will show later, we need the `timm` library so let's install it now.

**Tip**: Do all the installs at the start. If you install later in the notebook, you will need to restart the runtime and run all the cells again.

In [ ]:
!pip install -q timm                   # need this for object detection

## Image Classification Example

For an amazing variety of tasks, you can use pretrained models from the Hub and the `pipeline` function from the `transformers` library to do inference with just a few lines of code.

Let's first import `pipeline`.

In [ ]:
from transformers import pipeline

We next instantiate `pipeline` with the task we want to perform.

In [ ]:
pipe = pipeline("image-classification")

That's it - the pipeline is ready!

Let's feed an image to it. We will grab an image from the 'famous' COCO dataset.

In [ ]:
from PIL import Image
import requests

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

In [ ]:
image

Running it through the model is as simple as calling `pipe` with the image as the argument.

In [ ]:
pipe(image)

Looks like the category 'Egyptian cat' is predicted to be the right answer with a 93.7% probability.

BTW, you don't have to go with the **default** model HF has selected for this task. We can go to the [Hub](https://huggingface.co/models) and pick any 'image classification' model.

In [ ]:
pipe = pipeline("image-classification", model="microsoft/resnet-50")

In [ ]:
pipe(image)

The resnet-50 models predicts 'tiger cat' with a 94.1% probability.

## Object detection Example

OK, let's now try to detect all the objects in the picture.

In [ ]:
object_detection_pipe = pipeline("object-detection")

In [ ]:
results = object_detection_pipe(image)
results

Let's visualize the results:

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# from https://github.com/NielsRogge/Transformers-Tutorials/blob/master/HuggingFace_vision_ecosystem_overview_(June_2022).ipynb).

# colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125], [0.494, 0.184, 0.556], [0.466, 0.674, 0.188]]

def plot_results(image, results):
    plt.figure(figsize=(16,10))
    plt.imshow(image)
    ax = plt.gca()
    colors = COLORS * 100
    for result, color in zip(results, colors):
        box = result['box']
        xmin, xmax, ymin, ymax = box['xmin'], box['xmax'], box['ymin'], box['ymax']
        label = result['label']
        prob = result['score']
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=color, linewidth=3))
        text = f'{label}: {prob:0.2f}'
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor='yellow', alpha=0.5))
    plt.axis('off')
    plt.show()

In [ ]:
plot_results(image, results)

Pretty good, huh?

## Image segmentation Example



Instead of bounding boxes around each object, what if we want to get the exact shape of each object, at the pixel level?

In [ ]:
segmentation_pipe = pipeline("image-segmentation")

In [ ]:
result = segmentation_pipe(image)

In [ ]:
result

In [ ]:
result[0]['mask']

We can overlay this mask on the original image.

In [ ]:
plt.imshow(image)
plt.imshow(result[0]['mask'], alpha=0.7)

In [ ]:
plt.imshow(image)
plt.imshow(result[1]['mask'], alpha=0.7)

In [ ]:
plt.imshow(image)
plt.imshow(result[2]['mask'], alpha=0.7)

In [ ]:
plt.imshow(image)
plt.imshow(result[4]['mask'], alpha=0.7)

In [ ]:
plt.imshow(image)
plt.imshow(result[3]['mask'], alpha=0.7)

Notice that it **missed** the center part of the couch between the cats.